# Sumarização de Textos com PLN: Algoritmos via Biblioteca Sumy

**Autor: Wellington M Santos - Data Scientist**

[![LinkedIn](https://img.shields.io/badge/LinkedIn-wellington--moreira--santos-blue)](https://www.linkedin.com/in/wellington-moreira-santos)
[![Email](https://img.shields.io/badge/Email-wsantos08%40hotmail.com-red)](mailto:wsantos08@hotmail.com)


---

## 1. Introdução

Nos três projetos anteriores desta série, implementei algoritmos de sumarização do zero: frequência de palavras, algoritmo de Luhn e similaridade do cosseno com PageRank. Esse caminho foi intencional porque entender a mecânica interna de cada método é o que permite escolher, ajustar e diagnosticar um pipeline de sumarização em produção.

Neste projeto, o percurso é diferente. Uso a biblioteca `sumy`, que oferece seis algoritmos de sumarização prontos para uso, cada um com uma abordagem distinta. A ideia não é apenas aplicar funções de alto nível, mas entender o que cada algoritmo faz por baixo, conectar com o que já implementei, e comparar o desempenho de todos via ROUGE sobre o mesmo artigo e o mesmo resumo de referência.

Os seis algoritmos disponíveis na `sumy` cobrem um espectro amplo de abordagens: desde variantes dos métodos que já conheço (Luhn, TextRank) até técnicas de álgebra linear (LSA) e teoria da informação (KL-Sum). Ao final, uma tabela ROUGE ranqueia todos eles e fecha a série com uma perspectiva comparativa completa.

**Referência:** [sumy — PyPI](https://pypi.org/project/sumy/)


**Stack utilizada:** `Python 3.10+`, `nltk`, `sumy`, `newspaper4k`, `rouge-score`, `IPython.display`

**Seções:**

1. Introdução
2. Instalação e Configuração
3. Extração de Texto da Web
4. Estrutura da sumy
5. LuhnSummarizer
6. LsaSummarizer
7. LexRankSummarizer
8. TextRankSummarizer
9. SumBasicSummarizer
10. KLSummarizer
11. ReductionSummarizer
12. Comparação Geral com ROUGE
13. Conclusão e Considerações Finais



---


## 2. Instalação e Configuração


In [1]:
# Instalar dependências (executar uma vez)
# !pip install sumy newspaper4k rouge-score nltk

In [2]:
import nltk
from IPython.display import HTML, display

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\wsant\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\wsant\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\wsant\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [3]:
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.luhn import LuhnSummarizer
from sumy.summarizers.lsa import LsaSummarizer
from sumy.summarizers.lex_rank import LexRankSummarizer
from sumy.summarizers.text_rank import TextRankSummarizer
from sumy.summarizers.sum_basic import SumBasicSummarizer
from sumy.summarizers.kl import KLSummarizer
from sumy.summarizers.reduction import ReductionSummarizer


A `sumy` organiza o pipeline em três componentes independentes: um parser que ingere o texto, um tokenizador que conhece as regras linguísticas do idioma, e um sumarizador que aplica o algoritmo escolhido. Essa separação permite trocar qualquer componente sem alterar os outros.


---

## 3. Extração de Texto da Web


In [4]:
# !pip install newspaper4k
from newspaper import Article

In [5]:
def extrair_artigo(url, idioma='pt'):
    """
    Extrai título e texto principal de uma URL usando newspaper4k.

    Parâmetros:
        url (str): endereço do artigo
        idioma (str): código do idioma para o parser (padrão: 'pt')

    Retorna:
        tuple: (titulo, texto) ou (None, None) em caso de erro
    """
    try:
        artigo = Article(url, language=idioma)
        artigo.download()
        artigo.parse()
        return artigo.title, artigo.text
    except Exception as e:
        print(f"Erro ao extrair {url}: {e}")
        return None, None


In [7]:
url_pt = 'https://agenciabrasil.ebc.com.br/economia/noticia/2024-01/fmi-inteligencia-artificial-afetara-40-dos-empregos-em-todo-o-mundo'
titulo_pt, texto_pt = extrair_artigo(url_pt, idioma='pt')

print(f"Título: {titulo_pt}")
print(f"Tamanho: {len(texto_pt)} caracteres")
print(f"\nPrimeiros 300 caracteres:\n{texto_pt[:300]}...")

Título: FMI: inteligência artificial afetará 40% dos empregos em todo o mundo
Tamanho: 2124 caracteres

Primeiros 300 caracteres:
O desenvolvimento da inteligência artificial (IA) terá consequências para 40% dos empregos em todo o mundo, sobretudo nas economias avançadas, disse a diretora-geral do Fundo Monetário Internacional (FMI).

"No mundo, 40% dos empregos serão afetados. E mais: será o caso de quanto mais qualificado fo...



Uso o mesmo artigo do FMI sobre IA e empregos que utilizei nos projetos anteriores. Isso torna os resultados ROUGE desta série diretamente comparáveis entre si, já que o artigo e o resumo de referência são os mesmos.


---



## 4. Estrutura da sumy

Antes de aplicar cada algoritmo, vale entender como a `sumy` estrutura o pipeline. O `PlaintextParser` recebe o texto bruto e o `Tokenizer` com o idioma correspondente. O resultado é um objeto `document` que o sumarizador sabe como processar. A chamada final ao sumarizador recebe o documento e o número de sentenças desejadas no resumo.


In [8]:
# Estrutura base que se repete para todos os algoritmos
parser = PlaintextParser.from_string(texto_pt, Tokenizer('portuguese'))

# O objeto document contém as sentenças já tokenizadas
print(f"Total de sentenças no documento: {len(list(parser.document.sentences))}")

Total de sentenças no documento: 15


In [9]:
def formata_resumo(resumo_sumy):
    """
    Converte a saída do sumarizador sumy em uma lista de strings.

    Parâmetros:
        resumo_sumy: objeto retornado pelo sumarizador sumy

    Retorna:
        list[str]: lista de sentenças do resumo
    """
    return [str(sentenca) for sentenca in resumo_sumy]


In [10]:
def visualiza_resumo(titulo, texto_original, resumo):
    """
    Exibe o texto original com as sentenças do resumo destacadas.
    Suporta Jupyter (HTML) e outros ambientes (texto puro).

    Parâmetros:
        titulo (str): título exibido no cabeçalho
        texto_original (str): texto completo do artigo
        resumo (list[str]): sentenças selecionadas para o resumo
    """
    sentencas_originais = nltk.sent_tokenize(texto_original, language='portuguese')
    try:
        get_ipython  # noqa
        texto_html = ''
        for sentenca in sentencas_originais:
            if sentenca in resumo:
                texto_html += f'<mark>{sentenca}</mark> '
            else:
                texto_html += sentenca + ' '
        display(HTML(f'<h3>Resumo: {titulo}</h3><p>{texto_html}</p>'))
    except NameError:
        print(f"\n=== Resumo: {titulo} ===")
        for sentenca in sentencas_originais:
            marcador = ">> " if sentenca in resumo else "   "
            print(f"{marcador}{sentenca}")



---



## 5. LuhnSummarizer

O `LuhnSummarizer` da `sumy` implementa o mesmo algoritmo de Luhn que construí do zero no projeto anterior: identifica as palavras mais frequentes do texto, localiza seus clusters dentro de cada sentença e pontua cada sentença pela densidade do melhor cluster encontrado. Comparar os resultados desta implementação com a minha permite verificar se a implementação manual estava correta.


In [11]:

parser = PlaintextParser.from_string(texto_pt, Tokenizer('portuguese'))
sumarizador = LuhnSummarizer()
resumo_luhn = formata_resumo(sumarizador(parser.document, 5))

print(f"Sentenças selecionadas pelo LuhnSummarizer:")
for i, s in enumerate(resumo_luhn, 1):
    print(f"  [{i}] {s}")

Sentenças selecionadas pelo LuhnSummarizer:
  [1] O desenvolvimento da inteligência artificial (IA) terá consequências para 40% dos empregos em todo o mundo, sobretudo nas economias avançadas, disse a diretora-geral do Fundo Monetário Internacional (FMI).
  [2] Portanto, para as economias avançadas e alguns países emergentes, 60% dos empregos serão afetados", declarou Kristalina Georgieva.
  [3] "É certo que haverá impacto" disse Georgieva, observando que a IA pode acabar com alguns empregos e melhorar outros.
  [4] De acordo com o relatório, Singapura, Estados Unidos e Canadá são os países que estão melhor preparados até agora para a integração da IA.
  [5] "Devemos concentrar-nos nos países de rendimento mais baixo", destacou a diretora-geral do FMI, que demonstrou receio com o risco de abandono escolar nos Estados mais pobres.


In [12]:

visualiza_resumo(f"{titulo_pt} — Luhn", texto_pt, resumo_luhn)



---



## 6. LsaSummarizer

O `LsaSummarizer` usa Análise Semântica Latente (LSA), uma técnica de álgebra linear que representa o texto como uma matriz sentenças-por-termos e aplica Decomposição em Valores Singulares (SVD). O SVD identifica os "temas latentes" do texto, ou seja, padrões de co-ocorrência de palavras que não aparecem explicitamente mas emergem da estrutura matemática da matriz. As sentenças que melhor representam esses temas recebem pontuação mais alta.

Em termos práticos, o LSA é mais robusto que os métodos baseados em frequência simples em textos onde um mesmo conceito é expresso com vocabulário variado, porque capta a estrutura temática subjacente em vez de depender da repetição literal de palavras.


In [13]:
parser = PlaintextParser.from_string(texto_pt, Tokenizer('portuguese'))
sumarizador = LsaSummarizer()
resumo_lsa = formata_resumo(sumarizador(parser.document, 5))

print(f"Sentenças selecionadas pelo LsaSummarizer:")
for i, s in enumerate(resumo_lsa, 1):
    print(f"  [{i}] {s}")

Sentenças selecionadas pelo LsaSummarizer:
  [1] O desenvolvimento da inteligência artificial (IA) terá consequências para 40% dos empregos em todo o mundo, sobretudo nas economias avançadas, disse a diretora-geral do Fundo Monetário Internacional (FMI).
  [2] Portanto, para as economias avançadas e alguns países emergentes, 60% dos empregos serão afetados", declarou Kristalina Georgieva.
  [3] Os dados são de relatório divulgado pelo FMI antes das reuniões do Fórum Económico Mundial em Davos, que começam nesta segunda-feira (15) na estância alpina suíça.
  [4] De acordo com o relatório, Singapura, Estados Unidos e Canadá são os países que estão melhor preparados até agora para a integração da IA.
  [5] "Devemos concentrar-nos nos países de rendimento mais baixo", destacou a diretora-geral do FMI, que demonstrou receio com o risco de abandono escolar nos Estados mais pobres.


In [14]:
visualiza_resumo(f"{titulo_pt} — LSA", texto_pt, resumo_lsa)


---



## 7. LexRankSummarizer

O `LexRankSummarizer` implementa o algoritmo LexRank, que usa a mesma arquitetura de grafo e PageRank que implementei no projeto de similaridade do cosseno. A diferença está na forma como a similaridade entre sentenças é calculada: o LexRank usa TF-IDF (Term Frequency-Inverse Document Frequency) em vez de frequência bruta de palavras. O IDF penaliza palavras que aparecem em muitas sentenças do texto, reduzindo o peso de termos comuns e ampliando o peso de termos específicos. O resultado é uma medida de similaridade mais discriminativa que a do cosseno com frequência simples.


In [15]:
parser = PlaintextParser.from_string(texto_pt, Tokenizer('portuguese'))
sumarizador = LexRankSummarizer()
resumo_lexrank = formata_resumo(sumarizador(parser.document, 5))

print(f"Sentenças selecionadas pelo LexRankSummarizer:")
for i, s in enumerate(resumo_lexrank, 1):
    print(f"  [{i}] {s}")

Sentenças selecionadas pelo LexRankSummarizer:
  [1] Portanto, para as economias avançadas e alguns países emergentes, 60% dos empregos serão afetados", declarou Kristalina Georgieva.
  [2] Em entrevista à agência de notícias France-Presse (AFP), ela explicou que os impactos não são necessariamente negativos, pois também podem resultar em "aumento dos rendimentos".
  [3] A diretora defendeu que a prioridade deve ser ajudar os trabalhadores afetados e "partilhar os ganhos de produtividade".
  [4] "Devemos concentrar-nos nos países de rendimento mais baixo", destacou a diretora-geral do FMI, que demonstrou receio com o risco de abandono escolar nos Estados mais pobres.
  [5] "A IA pode ser assustadora, mas também pode ser uma grande oportunidade para todos", observou.


In [16]:
visualiza_resumo(f"{titulo_pt} — LexRank", texto_pt, resumo_lexrank)


---


## 8. TextRankSummarizer

O `TextRankSummarizer` implementa o algoritmo TextRank, que é a abordagem mais próxima do que implementei no projeto de similaridade do cosseno: grafo de similaridade entre sentenças com PageRank para identificar as mais centrais. A diferença em relação ao LexRank está no cálculo de similaridade: o TextRank usa sobreposição de palavras normalizada pelo comprimento das sentenças, enquanto o LexRank usa TF-IDF.

In [17]:
parser = PlaintextParser.from_string(texto_pt, Tokenizer('portuguese'))
sumarizador = TextRankSummarizer()
resumo_textrank = formata_resumo(sumarizador(parser.document, 5))

print(f"Sentenças selecionadas pelo TextRankSummarizer:")
for i, s in enumerate(resumo_textrank, 1):
    print(f"  [{i}] {s}")

Sentenças selecionadas pelo TextRankSummarizer:
  [1] O documento alerta que a IA poderá agravar as desigualdades salariais, prejudicando sobretudo a classe média, enquanto os trabalhadores com rendimentos já elevados poderão ver os seus salários "aumentarem mais do que a proporção" dos ganhos de produtividade com essa tecnologia.
  [2] A diretora defendeu que a prioridade deve ser ajudar os trabalhadores afetados e "partilhar os ganhos de produtividade".
  [3] De acordo com o relatório, Singapura, Estados Unidos e Canadá são os países que estão melhor preparados até agora para a integração da IA.
  [4] "Devemos concentrar-nos nos países de rendimento mais baixo", destacou a diretora-geral do FMI, que demonstrou receio com o risco de abandono escolar nos Estados mais pobres.
  [5] Segundo ela, em um contexto de abrandamento do ritmo de crescimento global, precisa-se "desesperadamente" de elementos capazes de aumentar a produtividade.


In [18]:
visualiza_resumo(f"{titulo_pt} — TextRank", texto_pt, resumo_textrank)


---


## 9. SumBasicSummarizer

O `SumBasicSummarizer` implementa o algoritmo SumBasic, que tem uma lógica iterativa diferente de todos os anteriores. Em vez de pontuar todas as sentenças de uma vez e selecionar as melhores, ele funciona em rodadas: seleciona a sentença com maior probabilidade de conter palavras frequentes, e depois reduz o peso dessas palavras para penalizar sentenças redundantes nas rodadas seguintes. O objetivo é diversificar o conteúdo do resumo, evitando que duas sentenças selecionadas falem essencialmente da mesma coisa.


In [19]:
parser = PlaintextParser.from_string(texto_pt, Tokenizer('portuguese'))
sumarizador = SumBasicSummarizer()
resumo_sumbasic = formata_resumo(sumarizador(parser.document, 5))

print(f"Sentenças selecionadas pelo SumBasicSummarizer:")
for i, s in enumerate(resumo_sumbasic, 1):
    print(f"  [{i}] {s}")

Sentenças selecionadas pelo SumBasicSummarizer:
  [1] "No mundo, 40% dos empregos serão afetados.
  [2] E mais: será o caso de quanto mais qualificado for o emprego.
  [3] A diretora defendeu que a prioridade deve ser ajudar os trabalhadores afetados e "partilhar os ganhos de produtividade".
  [4] "Devemos concentrar-nos nos países de rendimento mais baixo", destacou a diretora-geral do FMI, que demonstrou receio com o risco de abandono escolar nos Estados mais pobres.
  [5] "A IA pode ser assustadora, mas também pode ser uma grande oportunidade para todos", observou.


In [20]:

visualiza_resumo(f"{titulo_pt} — SumBasic", texto_pt, resumo_sumbasic)


---



## 10. KLSummarizer

O `KLSummarizer` usa a divergência de Kullback-Leibler, uma medida originada na teoria da informação. A ideia é selecionar o conjunto de sentenças cuja distribuição de palavras seja a mais próxima possível da distribuição de palavras do texto original. Em termos práticos, o resumo gerado pelo KL-Sum deve "soar" como uma versão comprimida do texto inteiro, no sentido de que as proporções com que os termos aparecem no resumo se aproximam das proporções do documento completo.


In [21]:

parser = PlaintextParser.from_string(texto_pt, Tokenizer('portuguese'))
sumarizador = KLSummarizer()
resumo_kl = formata_resumo(sumarizador(parser.document, 5))

print(f"Sentenças selecionadas pelo KLSummarizer:")
for i, s in enumerate(resumo_kl, 1):
    print(f"  [{i}] {s}")

Sentenças selecionadas pelo KLSummarizer:
  [1] O desenvolvimento da inteligência artificial (IA) terá consequências para 40% dos empregos em todo o mundo, sobretudo nas economias avançadas, disse a diretora-geral do Fundo Monetário Internacional (FMI).
  [2] "No mundo, 40% dos empregos serão afetados.
  [3] E mais: será o caso de quanto mais qualificado for o emprego.
  [4] Segundo ela, em um contexto de abrandamento do ritmo de crescimento global, precisa-se "desesperadamente" de elementos capazes de aumentar a produtividade.
  [5] "A IA pode ser assustadora, mas também pode ser uma grande oportunidade para todos", observou.


In [22]:

visualiza_resumo(f"{titulo_pt} — KL-Sum", texto_pt, resumo_kl)


---



## 11. ReductionSummarizer

O `ReductionSummarizer` aborda o problema de sumarização como um problema de compressão de grafo. Constrói um grafo onde as sentenças são nós e as arestas representam conexões baseadas em palavras compartilhadas. Em seguida, remove iterativamente os nós menos conectados, ou seja, as sentenças com menos relações com o restante do texto, até atingir o tamanho de resumo desejado. O resultado são as sentenças que formam o núcleo mais interconectado do texto.



In [23]:
parser = PlaintextParser.from_string(texto_pt, Tokenizer('portuguese'))
sumarizador = ReductionSummarizer()
resumo_reduction = formata_resumo(sumarizador(parser.document, 5))

print(f"Sentenças selecionadas pelo ReductionSummarizer:")
for i, s in enumerate(resumo_reduction, 1):
    print(f"  [{i}] {s}")


Sentenças selecionadas pelo ReductionSummarizer:
  [1] O documento alerta que a IA poderá agravar as desigualdades salariais, prejudicando sobretudo a classe média, enquanto os trabalhadores com rendimentos já elevados poderão ver os seus salários "aumentarem mais do que a proporção" dos ganhos de produtividade com essa tecnologia.
  [2] A diretora defendeu que a prioridade deve ser ajudar os trabalhadores afetados e "partilhar os ganhos de produtividade".
  [3] De acordo com o relatório, Singapura, Estados Unidos e Canadá são os países que estão melhor preparados até agora para a integração da IA.
  [4] "Devemos concentrar-nos nos países de rendimento mais baixo", destacou a diretora-geral do FMI, que demonstrou receio com o risco de abandono escolar nos Estados mais pobres.
  [5] Segundo ela, em um contexto de abrandamento do ritmo de crescimento global, precisa-se "desesperadamente" de elementos capazes de aumentar a produtividade.


In [24]:
visualiza_resumo(f"{titulo_pt} — Reduction", texto_pt, resumo_reduction)


---



## 12. Comparação Geral com ROUGE

Com os seis algoritmos aplicados sobre o mesmo artigo, posso ranqueá-los objetivamente. Uso o mesmo resumo de referência dos projetos anteriores, mantendo a comparabilidade de toda a série.


In [25]:
# !pip install rouge-score
from rouge_score import rouge_scorer

In [26]:
def avaliar_rouge(resumo_gerado, resumo_referencia):
    """
    Calcula métricas ROUGE entre o resumo gerado e o de referência.

    Parâmetros:
        resumo_gerado (list[str] ou str): resumo produzido pelo algoritmo
        resumo_referencia (str): resumo de referência escrito por humano

    Retorna:
        dict: scores ROUGE-1, ROUGE-2 e ROUGE-L
    """
    if isinstance(resumo_gerado, list):
        resumo_gerado = ' '.join(resumo_gerado)
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)
    return scorer.score(resumo_referencia.strip(), resumo_gerado.strip())

In [27]:
# Resumo de referência escrito manualmente
# Fonte: https://agenciabrasil.ebc.com.br/economia/noticia/2024-01/fmi-inteligencia-artificial-afetara-40-dos-empregos-em-todo-o-mundo
resumo_referencia = """
O FMI alerta que a inteligência artificial afetará 40% dos empregos em todo o mundo,
com impacto ainda maior nas economias avançadas, onde 60% dos postos de trabalho
serão impactados. A diretora-geral do fundo ressalta que os efeitos não são
necessariamente negativos e podem resultar em aumento de rendimentos, mas alerta
para o risco de aprofundamento das desigualdades entre países com diferentes
capacidades de adaptação tecnológica.
"""

In [28]:
algoritmos = {
    'Luhn':      resumo_luhn,
    'LSA':       resumo_lsa,
    'LexRank':   resumo_lexrank,
    'TextRank':  resumo_textrank,
    'SumBasic':  resumo_sumbasic,
    'KL-Sum':    resumo_kl,
    'Reduction': resumo_reduction,
}

scores = {nome: avaliar_rouge(resumo, resumo_referencia) for nome, resumo in algoritmos.items()}
scores

{'Luhn': {'rouge1': Score(precision=0.3488372093023256, recall=0.6, fmeasure=0.4411764705882353),
  'rouge2': Score(precision=0.171875, recall=0.2972972972972973, fmeasure=0.21782178217821785),
  'rougeL': Score(precision=0.23255813953488372, recall=0.4, fmeasure=0.29411764705882354)},
 'LSA': {'rouge1': Score(precision=0.3219178082191781, recall=0.6266666666666667, fmeasure=0.42533936651583715),
  'rouge2': Score(precision=0.14482758620689656, recall=0.28378378378378377, fmeasure=0.1917808219178082),
  'rougeL': Score(precision=0.21232876712328766, recall=0.41333333333333333, fmeasure=0.28054298642533937)},
 'LexRank': {'rouge1': Score(precision=0.3805309734513274, recall=0.5733333333333334, fmeasure=0.4574468085106383),
  'rouge2': Score(precision=0.1875, recall=0.28378378378378377, fmeasure=0.22580645161290322),
  'rougeL': Score(precision=0.22123893805309736, recall=0.3333333333333333, fmeasure=0.26595744680851063)},
 'TextRank': {'rouge1': Score(precision=0.23776223776223776, reca

In [29]:
print("Comparação ROUGE — Algoritmos sumy")
print("-" * 58)
print(f"{'Algoritmo':12} {'ROUGE-1 F1':>12} {'ROUGE-2 F1':>12} {'ROUGE-L F1':>12}")
print("-" * 58)

for nome, s in sorted(scores.items(), key=lambda x: x[1]['rouge1'].fmeasure, reverse=True):
    r1 = s['rouge1'].fmeasure
    r2 = s['rouge2'].fmeasure
    rl = s['rougeL'].fmeasure
    print(f"{nome:12} {r1:>12.3f} {r2:>12.3f} {rl:>12.3f}")

print("-" * 58)
melhor_r1 = max(scores, key=lambda x: scores[x]['rouge1'].fmeasure)
melhor_r2 = max(scores, key=lambda x: scores[x]['rouge2'].fmeasure)
melhor_rl = max(scores, key=lambda x: scores[x]['rougeL'].fmeasure)
print(f"{'Melhor':12} {melhor_r1:>12} {melhor_r2:>12} {melhor_rl:>12}")

Comparação ROUGE — Algoritmos sumy
----------------------------------------------------------
Algoritmo      ROUGE-1 F1   ROUGE-2 F1   ROUGE-L F1
----------------------------------------------------------
LexRank             0.457        0.226        0.266
Luhn                0.441        0.218        0.294
LSA                 0.425        0.192        0.281
KL-Sum              0.422        0.195        0.313
SumBasic            0.367        0.128        0.203
TextRank            0.312        0.083        0.183
Reduction           0.312        0.083        0.183
----------------------------------------------------------
Melhor            LexRank      LexRank       KL-Sum


O LexRank liderou em ROUGE-1 e ROUGE-2, e o KL-Sum foi o melhor em ROUGE-L. Alguns pontos merecem atenção.

O LexRank superou o TextRank apesar de ambos usarem grafos de similaridade com PageRank. A diferença está na métrica de similaridade: o LexRank usa TF-IDF, que penaliza palavras comuns e amplifica o peso de termos específicos do texto, produzindo arestas mais discriminativas no grafo. O TextRank usa sobreposição normalizada de palavras, uma medida mais simples e mais suscetível ao ruído de termos frequentes.

O KL-Sum, embora quarto colocado em ROUGE-1, obteve o melhor ROUGE-L (0.313), que mede a maior subsequência comum entre o resumo gerado e o de referência. Isso sugere que o KL-Sum tende a selecionar sentenças com estrutura sequencial mais parecida com a de um resumo escrito por humano, coerente com sua premissa: minimizar a divergência entre a distribuição de palavras do resumo e a do texto original produz um resumo que "soa" como o documento inteiro comprimido.

TextRank e Reduction empataram exatamente em todas as métricas (0.312, 0.083, 0.183), o que indica que, para este artigo específico, os dois algoritmos selecionaram o mesmo conjunto de sentenças. Isso não é necessariamente esperado em geral, mas faz sentido quando o texto tem uma estrutura de grafo simples o suficiente para que a remoção iterativa de nós periféricos (Reduction) e a centralidade por PageRank com sobreposição simples (TextRank) convirjam para o mesmo resultado.

Como nos projetos anteriores, esses valores são específicos para este artigo e para o resumo de referência utilizado. A ordem entre os algoritmos pode variar com textos de estrutura temática diferente.



---



## 13. Conclusão e Considerações Finais

### 13.1 Síntese do Projeto

Neste projeto apliquei seis algoritmos de sumarização disponíveis na biblioteca `sumy` sobre o mesmo artigo usado nos projetos anteriores da série, e avaliei todos via ROUGE com o mesmo resumo de referência. Na comparação, o LexRank liderou em ROUGE-1 (0.457) e ROUGE-2 (0.226), enquanto o KL-Sum obteve o melhor ROUGE-L (0.313). TextRank e Reduction empataram em todas as métricas (0.312, 0.083, 0.183), sugerindo que para este artigo os dois algoritmos convergiram para o mesmo conjunto de sentenças. Cada algoritmo representa uma abordagem distinta: o Luhn usa clusters de proximidade de palavras frequentes; o LSA usa decomposição matricial para identificar temas latentes; o LexRank e o TextRank usam grafos de similaridade com PageRank, diferindo na métrica de similaridade (TF-IDF versus sobreposição normalizada); o SumBasic usa seleção iterativa com penalização de redundância; o KL-Sum minimiza a divergência entre a distribuição de palavras do resumo e a do texto original; e o Reduction remove iterativamente as sentenças menos conectadas no grafo do texto.

### 13.2 sumy vs. Implementações do Zero

Usar a `sumy` tem vantagens claras em produção: código conciso, suporte a múltiplos idiomas via `Tokenizer`, e algoritmos testados por uma comunidade ativa. A desvantagem é a menor transparência: ao chamar `LuhnSummarizer()`, não é imediato o que acontece internamente com os hiperparâmetros de distância e `top_n`. Os três projetos anteriores existem exatamente para preencher essa lacuna: quem implementou Luhn do zero consegue inspecionar e ajustar o comportamento da `sumy` com muito mais clareza do que quem chegou diretamente à biblioteca.

### 13.3 Considerações Finais sobre a Série

Esta série percorre uma progressão completa em sumarização extrativa com PLN clássico. Partindo da contagem simples de palavras, passando por clusters de proximidade, grafos de similaridade, e chegando a técnicas de álgebra linear e teoria da informação, cada projeto adicionou uma camada de sofisticação sobre o anterior. Este último fecha o ciclo mostrando como as mesmas ideias são empacotadas em bibliotecas de produção e como avaliá-las de forma sistemática e comparável.



---



> Material de estudo desenvolvido para o repositório [data-trivium](https://github.com/wellington-moreira-santos/data-trivium), a partir do curso de Sumarização de Textos com PLN da IAExpert Academy.  
